# Praktikum 3: Approximate Nearest Neighbor Menggunakan HNSWlib

In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679801 sha256=c2dc63595cdb01a5cd8594e8c0430fa8454e77bbcf6c9f739614be498fa4f59e
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# 1. Buat data 2D acak
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# 2. Exact NN (Brute Force)
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", round((end - start) * 1000, 4), "ms")

# 3. HNSW
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances_hnsw = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances_hnsw)
print("Waktu:", round((end - start) * 1000, 4), "ms")

=== Exact NN ===
Indices: [[484   8 513 413 360]]
Distances: [[0.01694982 0.01898158 0.02194151 0.0296094  0.04342842]]
Waktu: 103.8949 ms

=== HNSW ===
Indices: [[484   8 513 413 360]]
Distances: [[0.0002873  0.0003603  0.00048143 0.00087672 0.00188603]]
Waktu: 0.2844 ms


In [3]:
def run_hnsw_experiment(metric, num_elements, dim, k=5):
    np.random.seed(42)
    data_exp = np.random.random((num_elements, dim)).astype(np.float32)
    query_exp = np.array([[0.5] * dim], dtype=np.float32)

    # Map metric name to Scikit-learn and HNSWlib spaces
    if metric == 'Euclidean (L2)':
        sk_metric = 'euclidean'
        hnsw_space = 'l2'
    else:  # Cosine / IP
        sk_metric = 'cosine'
        hnsw_space = 'cosine'

    # 1. Exact NN
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=sk_metric)
    nn.fit(data_exp)
    t0 = time.time()
    distances_exact, indices_exact = nn.kneighbors(query_exp)
    time_exact = (time.time() - t0) * 1000

    # 2. HNSW
    p = hnswlib.Index(space=hnsw_space, dim=dim)
    p.init_index(max_elements=num_elements, ef_construction=100, M=16)
    p.add_items(data_exp)
    p.set_ef(50)

    t0 = time.time()
    labels_hnsw, distances_hnsw = p.knn_query(query_exp, k=k)
    time_hnsw = (time.time() - t0) * 1000

    print(f"Metric: {metric:14s} | Data: {num_elements:7d} | Dim: {dim}D | "
          f"Index Exact vs HNSW: {list(indices_exact[0])}, {list(labels_hnsw[0])} | "
          f"Waktu (ms): {time_exact:.4f} vs {time_hnsw:.4f}")

print("=== HASIL EKSPERIMEN TABEL HNSW ===")
run_hnsw_experiment('Euclidean (L2)', 1000, 2)
run_hnsw_experiment('Euclidean (L2)', 1000, 5)
run_hnsw_experiment('Euclidean (L2)', 1000000, 2)
run_hnsw_experiment('Euclidean (L2)', 1000000, 5)

run_hnsw_experiment('Cosine', 1000, 2)
run_hnsw_experiment('Cosine', 1000, 5)
run_hnsw_experiment('Cosine', 1000000, 2)
run_hnsw_experiment('Cosine', 1000000, 5)

=== HASIL EKSPERIMEN TABEL HNSW ===
Metric: Euclidean (L2) | Data:    1000 | Dim: 2D | Index Exact vs HNSW: [np.int64(137), np.int64(170), np.int64(750), np.int64(316), np.int64(991)], [np.uint64(137), np.uint64(170), np.uint64(750), np.uint64(316), np.uint64(991)] | Waktu (ms): 0.8669 vs 0.0873
Metric: Euclidean (L2) | Data:    1000 | Dim: 5D | Index Exact vs HNSW: [np.int64(895), np.int64(451), np.int64(375), np.int64(386), np.int64(411)], [np.uint64(895), np.uint64(451), np.uint64(375), np.uint64(386), np.uint64(411)] | Waktu (ms): 1.0545 vs 0.0908
Metric: Euclidean (L2) | Data: 1000000 | Dim: 2D | Index Exact vs HNSW: [np.int64(286942), np.int64(583487), np.int64(695123), np.int64(948064), np.int64(829288)], [np.uint64(286942), np.uint64(583487), np.uint64(695123), np.uint64(948064), np.uint64(829288)] | Waktu (ms): 66.9768 vs 0.1211
Metric: Euclidean (L2) | Data: 1000000 | Dim: 5D | Index Exact vs HNSW: [np.int64(794791), np.int64(501340), np.int64(912100), np.int64(84056), np.int